In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import cv2
import os


In [2]:
def white_balance(img):

    img = np.array(img).astype(float)

    ## average value of each channel
    avg_red = np.mean(img[:, :, 0])
    avg_green = np.mean(img[:, :, 1])
    avg_blue = np.mean(img[:, :, 2])
    
    # gray value calculation
    avg_gray = (avg_red + avg_green + avg_blue) / 3
    
    # Scaling each channel
    img[:, :, 0] *= (avg_gray / avg_red)
    img[:, :, 1] *= (avg_gray / avg_green)
    img[:, :, 2] *= (avg_gray / avg_blue)
    
    # Clip values to [0, 255]
    img = np.clip(img, 0, 255).astype(np.uint8)     #the factor can cause values to go beyond 255, hence bounding it.
    return Image.fromarray(img)


### CLAHE

In [3]:

def apply_clahe(img):
    img_np = np.array(img)
    lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l = clahe.apply(l)
    lab = cv2.merge((l,a,b))
    return Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB))

In [17]:
## replace Positive as 1 and Negative as 0
df = pd.read_csv('dataset_120.csv')
df = df.drop(columns=['patient_id'])
mapping_dict = {'Positive': 1, 'Negative': 0}
df['label'] = df['label'].map(mapping_dict)
df.to_csv('updated_dataset_120.csv', index=False)


resnet = models.resnet18(pretrained=True)
feature_extractor = nn.Sequential(*list(resnet.children())[:-1]) 
feature_extractor.eval()

def extract_image_features(image_path):
    preprocess = transforms.Compose([
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.4313, 0.3793, 0.4027], std=[0.1515, 0.1860, 0.1763]),
    ])
    img = Image.open(image_path).convert('RGB')
    img_t = preprocess(img).unsqueeze(0)
    
    with torch.no_grad():
        features = feature_extractor(img_t)
    return features.flatten().numpy()

# 2. Prepare Your Combined Dataset
# Replace these with your actual image paths and 7 symptom features
df = pd.read_csv("updated_dataset_120.csv") # ... 120 images
all_image_paths = []

for idx, path in enumerate(df['ImageName']):
    all_image_paths.append(os.path.join('data', df.iloc[idx, 0]))

# print(all_image_paths)
symptom_data =  (df.iloc[:, 2:9]).to_numpy()   # Replace with your (120, 7) symptoms array
# print(symptom_data)
labels = (df.iloc[:, 1]).to_numpy() # Replace with your actual labels
# print(labels)

# # Extract CNN features for all images
print("Extracting features from 120 images...")
image_features = np.array([extract_image_features(p) for p in all_image_paths])
print(image_features.shape)

# # 3. Combine CNN Features + Symptoms
X_combined = np.hstack((image_features, symptom_data))
print(X_combined.shape)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_combined)

# 4. Train with Leave-One-Out Cross-Validation (LOOCV)
# This is the gold standard for testing on very small datasets
loo = LeaveOneOut()
y_true, y_pred = [], []

print("Running Leave-One-Out validation (120 rounds)...")
for train_index, test_index in loo.split(X_scaled):
    X_train, X_test = X_scaled[train_index], X_scaled[test_index]
    y_train, y_test = labels[train_index], labels[test_index]
    
    # Use SVM with RBF kernel - excellent for small medical data
    clf = SVC(kernel='rbf', C=1.0, probability=True)
    clf.fit(X_train, y_train)
    
    y_true.append(y_test[0])
    y_pred.append(clf.predict(X_test)[0])

# 5. Evaluation
print("\nFinal Results:")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.2f}")
print(classification_report(y_true, y_pred))


c:\Users\avk1028\AppData\Local\anaconda3\envs\pytorch_env\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\avk1028\AppData\Local\anaconda3\envs\pytorch_env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Extracting features from 120 images...
(120, 512)
(120, 519)
Running Leave-One-Out validation (120 rounds)...

Final Results:
Accuracy: 0.57
              precision    recall  f1-score   support

           0       0.57      0.57      0.57        60
           1       0.57      0.57      0.57        60

    accuracy                           0.57       120
   macro avg       0.57      0.57      0.57       120
weighted avg       0.57      0.57      0.57       120

